In [1]:
# ===== 1: 설정 =====
# 참조 조문·inline 법령 정의는 04번과 같은 data/rag_context.py를 쓴다(그래프 5요소, limit 절단·중복 해결, citation_label).
# 임베딩 모델은 로드하지 않는다 — ContextBuilder는 Milvus 조회만 하므로 GPU는 생성 모델이 단독으로 쓴다.
import sys
import json
import re
import importlib
from pathlib import Path

PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
sys.path.insert(0, str(PROJECT_DIR))
FAQ_DIR = PROJECT_DIR / "data" / "faq"
FAQ_DIR.mkdir(parents=True, exist_ok=True)

import data.rag_context as rag_context
importlib.reload(rag_context)

builder = rag_context.ContextBuilder()      # 청크 JSON + 03번 그래프 + Milvus client (그래프 키 정합성 assert 포함)
chunks_dog = builder.chunks["강아지"]
chunks_cat = builder.chunks["고양이"]

print("강아지 청크:", len(chunks_dog), "/ 그래프 노드:", len(builder.graphs["강아지"]))
print("고양이 청크:", len(chunks_cat), "/ 그래프 노드:", len(builder.graphs["고양이"]))


강아지 청크: 928 / 그래프 노드: 446
고양이 청크: 850 / 그래프 노드: 408


In [2]:
# ===== 2: FAQ 생성 단위 만들기 (data/faq_units.py — 06번 품질 검증과 같은 단위 구성을 쓴다) =====
# 레코드를 '조(article_block)·등급' 단위로 FAQ_UNIT_MAX(1500자) 이하로 묶고, 법규는 빼고, 한 문장짜리 준용규정은 앞 단위에 붙인다.
# 규칙과 측정 근거는 모듈 주석 참고.
import data.faq_units as faq_units
importlib.reload(faq_units)
from data.faq_units import FAQ_UNIT_MAX, EXCLUDE_FROM_FAQ, unit_id, main_label, build_citation_suffix

units_dog = faq_units.make_units(chunks_dog)
units_cat = faq_units.make_units(chunks_cat)
for sp, units in [("강아지", units_dog), ("고양이", units_cat)]:
    lens = sorted(len(u["content"]) for u in units)
    print(f"{sp}: 단위 {len(units)}개 | 길이 중앙 {lens[len(lens)//2]} / 최대 {lens[-1]} | "
          f"FAQ_UNIT_MAX 초과(단일 레코드가 긴 표) {sum(l > FAQ_UNIT_MAX for l in lens)}")


강아지: 단위 248개 | 길이 중앙 715 / 최대 2470 | FAQ_UNIT_MAX 초과(단일 레코드가 긴 표) 6
고양이: 단위 221개 | 길이 중앙 646 / 최대 2470 | FAQ_UNIT_MAX 초과(단일 레코드가 긴 표) 5


In [3]:
# ===== 3: 조항형/안내형 분류 =====
# 조 번호가 있으면 조항형(보통약관·특별약관·기타_제도성·법규), 없으면 안내형(안내류·별표·별첨)

def get_prompt_type(unit: dict) -> str:
    return "조항형" if unit.get("article_num") is not None else "안내형"


In [4]:
# ===== 4: 참조 조회 (data/rag_context.py) =====
# 그래프 엣지의 ref_section으로 조회, 조·항 전체를 원문 순서로(limit 100), 중복 제거,
# inline 법령 박스는 03 규칙(【「법명, 제N조의M)으로 추출하고 citation_label(예: 소득세법 제59조의4(특별세액공제))을 단다.

def get_references_for_unit(unit: dict, species: str) -> list[dict]:
    return builder.resolve_references(unit, species, source_chunks=unit["records"])

# main_label(메인 출처 표기)은 data/faq_units.py (셀 2에서 import)


In [5]:
# ===== 5: 프롬프트 템플릿 =====
# V2(실험 C, 2026-10-14): 학습 질문의 19%가 '약관' 명칭을 담았다 — "출처:/조항:" 라벨이 질문으로 복사된 것.
# 라벨은 답변 정확도에 필요해서 그대로 두고, 라벨 바로 옆에 '답변 작성용'임을 적는다. v1은 기존과 바이트 단위로 같다.

ARTICLE_PROMPT_TEMPLATE = """출처: {section_name}
조항: {label}
{content}

관련 조항 (보충 설명용):
{references_text}
"""

GUIDE_PROMPT_TEMPLATE = """분류: {section} {section_name}
{content}
"""

ARTICLE_PROMPT_TEMPLATE_V2 = """출처(답변 작성용 — 질문에 쓰지 말 것): {section_name}
조항(답변 작성용 — 질문에 쓰지 말 것): {label}
{content}

관련 조항 (보충 설명용):
{references_text}
"""

GUIDE_PROMPT_TEMPLATE_V2 = """분류(답변 작성용 — 질문에 쓰지 말 것): {section} {section_name}
{content}
"""

def build_single_prompt(unit: dict, references: list[dict], version: str = "v1", topics: list[str] | None = None) -> str:
    """topics: 주제 지정(실험 C 안 A·B) — 끝에 [주제] 한 줄을 붙인다. 시스템 프롬프트가 이 줄을 가리킨다."""
    content = rag_context.display_text(unit)       # 뒷조각이면 [도입 문장/제목 경로]를 앞에 붙여 보여준다
    if get_prompt_type(unit) == "조항형":
        references_text = "".join(f"\n{ref['citation_label']}\n{ref['content']}\n"
                                  for ref in references if ref.get("content")) or "(없음)"
        template = ARTICLE_PROMPT_TEMPLATE_V2 if version in ("v2", "v2C") else ARTICLE_PROMPT_TEMPLATE
        prompt = template.format(
            section_name=unit.get("section_name") or unit["section"], label=main_label(unit),
            content=content, references_text=references_text)
    else:
        template = GUIDE_PROMPT_TEMPLATE_V2 if version in ("v2", "v2C") else GUIDE_PROMPT_TEMPLATE
        prompt = template.format(section=unit["section"], section_name=unit.get("section_name") or "", content=content)
    if topics:
        prompt += f"\n[주제] {', '.join(topics)}\n"
    return prompt

In [6]:
# ===== 6: Ollama 호출 (tri-7b:latest = Tri-7B Q4_K_M GGUF, docker 컨테이너 'ollama') =====
# 기존 FAQ와 같은 양자화·런타임을 유지한다. 컨테이너는 restart=no라 재부팅 후 `docker start ollama` 필요.
import requests
# 계산 금지 문구(위 시스템 프롬프트) — 2026-10-12 측정(의료비 48단위): 계산형 QA 오류 26% → 15%.
# 남는 오류는 생성 루프의 faq_verify 관문이 걸러낸다. 코드 계산표를 프롬프트에 넣는 방식은 오류 13%였지만
# 프롬프트가 길어져 JSON 파싱 실패가 4% → 31%로 늘어 채택하지 않았다.

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "tri-7b:latest"
NUM_CTX = 8192        # tri-7b Modelfile 값과 같게 명시(모델 자체는 32768까지 지원)
NUM_PREDICT = 1200
# 입력이 NUM_CTX - NUM_PREDICT를 넘으면 Ollama가 경고 없이 앞부분(시스템 프롬프트 포함)을 잘라낸다.
# 2026-10-12 측정(Tri-7B 토크나이저): 단위 698개 입력 중앙 608 / p90 1556 / 최대 2933토큰 — 초과 0. 그래도 응답마다 확인한다.
PROMPT_BUDGET = NUM_CTX - NUM_PREDICT
truncation_log = []   # (단위 식별, prompt_eval_count) — 예산에 닿은 호출
prompt_tokens_log = []   # (단위 식별, prompt_eval_count) — 모든 호출(실험 C에서 프롬프트 길이 실측)

SYSTEM_PROMPT_ARTICLE = """당신은 반려동물 보험 상담 FAQ를 만드는 전문가입니다.
- 주어진 약관 조항에 적힌 사실만 근거로 질문과 답변을 만드세요. 원문에 없는 숫자, 금액, 날짜, 비율은 절대로 만들어내지 마세요.
- 원문에 구체적인 수치가 없으면 "회사가 정한 기준에 따릅니다"처럼 돌려 표현하세요.
- 보험금 계산 질문은 원문에 있는 계산 예시(치료비·자기부담금·결과 금액)를 그대로 사용하세요. 원문에 없는 새 금액으로 직접 계산하지 마세요.
- 서로 다른 관점에서 3개의 질문-답변 쌍을 만드세요.
- 답변은 2~4문장으로 자연스럽게 설명하세요. "근거" 표기는 쓰지 마세요.
- 반드시 아래 JSON 배열 형식으로만 출력하세요.
[{"question": "...", "answer": "..."}, {"question": "...", "answer": "..."}, {"question": "...", "answer": "..."}]
"""

SYSTEM_PROMPT_GUIDE = """당신은 반려동물 보험 상담 FAQ를 만드는 전문가입니다.
- 주어진 안내 자료에 적힌 사실만 근거로 개념 설명형 질문과 답변을 만드세요. 지어내지 마세요.
- 서로 다른 관점에서 2개의 질문-답변 쌍을 만드세요.
- 답변은 2~4문장으로 자연스럽게 설명하세요. "근거" 표기는 쓰지 마세요.
- 반드시 아래 JSON 배열 형식으로만 출력하세요.
[{"question": "...", "answer": "..."}, {"question": "...", "answer": "..."}]
"""

# ----- 실험 C(2026-10-14): 질문 프롬프트 V2 -----
# 측정: 학습 질문 중 1인칭·상황형 1.1%(심판 통과·탈락 비율 같음 — 06이 아니라 05가 만들지 않았다), '약관' 명칭 19%.
# 바꾸는 것은 "서로 다른 관점에서 N개" 한 줄뿐이다. 답변 규칙(지어내기·계산 금지, 2~4문장, 근거 표기 금지)과
# JSON 형식 줄은 V1 문자열을 그대로 쓴다 — 아래 assert가 V1의 나머지 줄이 V2에 전부 남았는지 확인한다.
PROMPT_VERSION = "v1"     # "v2"면 V2 템플릿·시스템 프롬프트 (셀 9 전량 생성 기본값은 v1 그대로)
TOPIC_MODE = None         # 주제 지정: None | "A"(세 번째 질문 자리에 주제) | "B"(주제 전용 회차 1회 추가, 셀 10)

QUESTION_STYLE = ("- 질문은 이미 보험에 가입한 고객이 상담 창구에 묻는 말투로, 30자 내외 구어체로 쓰세요. "
                  "조문 번호, 약관 이름, 조항 제목은 질문에 쓰지 마세요.")
SITUATION_EXAMPLE = '자기 반려동물의 상황을 말하며 보장되는지 묻는 질문(예: "우리 아이가 ~했는데 보상되나요?")'
FALLBACK = "원문이 다루지 않는 성격이면 원문이 답하는 다른 질문으로 바꾸세요."

PERSPECTIVE_ARTICLE = "- 서로 다른 관점에서 3개의 질문-답변 쌍을 만드세요."
PERSPECTIVE_GUIDE = "- 서로 다른 관점에서 2개의 질문-답변 쌍을 만드세요."
GUIDE_FIRST_V1 = "- 주어진 안내 자료에 적힌 사실만 근거로 개념 설명형 질문과 답변을 만드세요. 지어내지 마세요."
GUIDE_FIRST_V2 = "- 주어진 안내 자료에 적힌 사실만 근거로 질문과 답변을 만드세요. 지어내지 마세요."   # '개념 설명형'만 뺀다
JSON_3 = SYSTEM_PROMPT_ARTICLE[SYSTEM_PROMPT_ARTICLE.index("- 반드시 아래 JSON"):]
JSON_2 = SYSTEM_PROMPT_GUIDE[SYSTEM_PROMPT_GUIDE.index("- 반드시 아래 JSON"):]

def _article_questions(third: str) -> str:
    return (f"{QUESTION_STYLE}\n- 질문 3개는 성격을 다르게 하세요: ① {SITUATION_EXAMPLE} "
            f"② 절차·기한·서류를 묻는 질문 ③ {third}. {FALLBACK}")

SYSTEM_PROMPT_ARTICLE_V2 = SYSTEM_PROMPT_ARTICLE.replace(PERSPECTIVE_ARTICLE, _article_questions("조건·범위를 확인하는 질문"))
# 안 A: 세 번째 자리에 주제 지정 — [주제] 줄은 build_single_prompt가 원문이 다루는 주제(faq_style.eligible_topics)만 붙인다
SYSTEM_PROMPT_ARTICLE_V2_TOPIC = SYSTEM_PROMPT_ARTICLE.replace(
    PERSPECTIVE_ARTICLE, _article_questions("아래 [주제] 중 이 원문이 답하는 것에 대해 고객이 가장 많이 묻는 질문"))
SYSTEM_PROMPT_GUIDE_V2 = SYSTEM_PROMPT_GUIDE.replace(GUIDE_FIRST_V1, GUIDE_FIRST_V2).replace(
    PERSPECTIVE_GUIDE,
    "- 질문은 보험에 가입한 고객이 상담 창구에 묻는 말투로, 30자 내외 구어체로 쓰세요. 안내 자료 이름이나 분류명은 질문에 쓰지 마세요.\n"
    f"- 질문 2개는 성격을 다르게 하세요: ① 자기 상황을 말하며 묻는 질문 ② 절차·조건을 확인하는 질문. {FALLBACK}")
# 안 B: 주제 전용 회차 — 조항형 답변 규칙(계산 금지 포함)을 그대로 두고 질문 지시와 개수(2개)만 바꾼다.
# 가입 나이는 가입 전 고객이 묻는 주제라 '이미 가입한 고객' 지시와 충돌한다 — 그 주제만 예외로 적는다.
SYSTEM_PROMPT_TOPIC = SYSTEM_PROMPT_ARTICLE.replace(
    PERSPECTIVE_ARTICLE,
    f"{QUESTION_STYLE}\n- 아래 [주제]에 대해 고객이 실제로 묻는 질문 2개를 만드세요. 1개는 {SITUATION_EXAMPLE}으로 하세요. "
    "원문이 답하는 범위 안에서만 만드세요. 가입 나이는 가입을 고민하는 고객이 묻는 질문이어도 됩니다.").replace(JSON_3, JSON_2)

# ----- 실험 C-2(2026-10-14): 질문 프롬프트 V2C -----
# V2는 질문 성질을 바꿨지만(라벨 누출 11.3% → 0~1.5%, 상황형 1.4% → 56~58%) 심판 통과율이 v1 76.1% → 36.5%(v2A)/37.9%(v2B r1).
# 탈락 QA를 읽은 원인 두 가지: ① 한 질문에 두 가지를 물어 incomplete(v2B r1 15건, 최다 탈락 사유), 질문 길이 중앙 56 → 65자
# ② 원문에 없는 설정(나이·계약 방식·질병명)을 지어내 fabricated(v2A 15건)·clause_mix(v2A·v2B 각 8건) 증가.
# V2 세 프롬프트(조항형·안내형·주제 회차)에 아래 두 줄만 더한다 — 다른 변경이 섞이면 원인 분리가 안 된다.
# 위치는 질문 지시 바로 뒤(답변 줄 앞). 안 A(V2_TOPIC)는 실험 C-2 설정에 없어서 만들지 않는다.
ONE_ASK = "- 한 질문에서는 한 가지만 물으세요. 질문 두 개를 이어 붙이지 마세요."
NO_NEW_SETTING = ("- 질문에 쓰는 상황은 원문이 다루는 범위 안에서만 만드세요. "
                  "원문에 없는 나이·계약 방식·질병명·금액 같은 설정을 새로 만들지 마세요.")
ANSWER_LINE = '- 답변은 2~4문장으로 자연스럽게 설명하세요. "근거" 표기는 쓰지 마세요.'

def _v2c(v2: str) -> str:
    assert v2.count(ANSWER_LINE) == 1, "답변 줄을 V2에서 찾지 못함"
    v2c = v2.replace(ANSWER_LINE, f"{ONE_ASK}\n{NO_NEW_SETTING}\n{ANSWER_LINE}")
    # V2 대비 추가된 것이 두 줄뿐인지 — 두 줄을 빼면 V2와 줄 순서까지 같아야 한다
    assert [l for l in v2c.splitlines() if l not in (ONE_ASK, NO_NEW_SETTING)] == v2.splitlines()
    return v2c

SYSTEM_PROMPT_ARTICLE_V2C = _v2c(SYSTEM_PROMPT_ARTICLE_V2)
SYSTEM_PROMPT_GUIDE_V2C = _v2c(SYSTEM_PROMPT_GUIDE_V2)
SYSTEM_PROMPT_TOPIC_V2C = _v2c(SYSTEM_PROMPT_TOPIC)

def _lines(s: str) -> set[str]:
    return {l for l in s.splitlines() if l.strip()}
for v1, v2, replaced in [(SYSTEM_PROMPT_ARTICLE, SYSTEM_PROMPT_ARTICLE_V2, {PERSPECTIVE_ARTICLE}),
                         (SYSTEM_PROMPT_ARTICLE, SYSTEM_PROMPT_ARTICLE_V2_TOPIC, {PERSPECTIVE_ARTICLE}),
                         (SYSTEM_PROMPT_GUIDE, SYSTEM_PROMPT_GUIDE_V2, {PERSPECTIVE_GUIDE, GUIDE_FIRST_V1}),
                         (SYSTEM_PROMPT_ARTICLE, SYSTEM_PROMPT_TOPIC, {PERSPECTIVE_ARTICLE} | _lines(JSON_3)),
                         (SYSTEM_PROMPT_ARTICLE, SYSTEM_PROMPT_ARTICLE_V2C, {PERSPECTIVE_ARTICLE}),
                         (SYSTEM_PROMPT_GUIDE, SYSTEM_PROMPT_GUIDE_V2C, {PERSPECTIVE_GUIDE, GUIDE_FIRST_V1}),
                         (SYSTEM_PROMPT_ARTICLE, SYSTEM_PROMPT_TOPIC_V2C, {PERSPECTIVE_ARTICLE} | _lines(JSON_3))]:
    assert v1 != v2 and replaced <= _lines(v1), "바꿀 줄을 V1에서 찾지 못함"
    assert _lines(v1) - replaced <= _lines(v2), f"V2에서 빠진 줄: {_lines(v1) - replaced - _lines(v2)}"
assert JSON_2 in SYSTEM_PROMPT_TOPIC and JSON_3 in SYSTEM_PROMPT_ARTICLE_V2 and JSON_2 in SYSTEM_PROMPT_GUIDE_V2
assert JSON_2 in SYSTEM_PROMPT_TOPIC_V2C and JSON_3 in SYSTEM_PROMPT_ARTICLE_V2C and JSON_2 in SYSTEM_PROMPT_GUIDE_V2C
for name, s in [("ARTICLE", SYSTEM_PROMPT_ARTICLE), ("ARTICLE_V2", SYSTEM_PROMPT_ARTICLE_V2),
                ("ARTICLE_V2_TOPIC", SYSTEM_PROMPT_ARTICLE_V2_TOPIC), ("GUIDE", SYSTEM_PROMPT_GUIDE),
                ("GUIDE_V2", SYSTEM_PROMPT_GUIDE_V2), ("TOPIC", SYSTEM_PROMPT_TOPIC),
                ("ARTICLE_V2C", SYSTEM_PROMPT_ARTICLE_V2C), ("GUIDE_V2C", SYSTEM_PROMPT_GUIDE_V2C),
                ("TOPIC_V2C", SYSTEM_PROMPT_TOPIC_V2C)]:
    print(f"SYSTEM_PROMPT_{name}: {len(s)}자")

tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
assert any(m["name"] == MODEL_NAME for m in tags["models"]), f"{MODEL_NAME} 없음 — `docker start ollama` 확인"
print("Ollama 모델 확인:", MODEL_NAME)

# unit_id(단위 식별자)는 data/faq_units.py (셀 2에서 import)

def call_ollama(prompt: str, unit: dict, temperature: float = 0.5, system: str | None = None) -> str:
    """system: 시스템 프롬프트를 직접 고를 때(V2·주제 회차). None이면 기존대로 유형별 V1"""
    system_prompt = system or (SYSTEM_PROMPT_ARTICLE if get_prompt_type(unit) == "조항형" else SYSTEM_PROMPT_GUIDE)
    response = requests.post(f"{OLLAMA_URL}/api/generate", json={
        "model": MODEL_NAME,
        "system": system_prompt,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "repeat_penalty": 1.1,
                    "num_predict": NUM_PREDICT, "num_ctx": NUM_CTX},
    })
    response.raise_for_status()
    body = response.json()
    used = body.get("prompt_eval_count") or 0
    prompt_tokens_log.append((unit_id(unit), used))
    if used >= PROMPT_BUDGET:
        truncation_log.append((unit_id(unit), used))
        print(f"  !! 입력 {used}토큰 ≥ 예산 {PROMPT_BUDGET} — 앞부분이 잘렸을 수 있음: {unit_id(unit)}")
    return body["response"]


SYSTEM_PROMPT_ARTICLE: 460자
SYSTEM_PROMPT_ARTICLE_V2: 666자
SYSTEM_PROMPT_ARTICLE_V2_TOPIC: 693자
SYSTEM_PROMPT_GUIDE: 270자
SYSTEM_PROMPT_GUIDE_V2: 406자
SYSTEM_PROMPT_TOPIC: 635자
SYSTEM_PROMPT_ARTICLE_V2C: 785자
SYSTEM_PROMPT_GUIDE_V2C: 525자
SYSTEM_PROMPT_TOPIC_V2C: 754자
Ollama 모델 확인: tri-7b:latest


In [7]:
# ===== 7: 생성 루프 (재시도 + 중국어 필터 + 근거 표기는 코드로 조립) =====
import time
import random
import data.faq_verify as faq_verify
import data.faq_style as faq_style      # 주제 정의(실험 C) — 비교 리포트와 같은 정규식
importlib.reload(faq_verify)
importlib.reload(faq_style)

def has_chinese_sentence(text: str) -> bool:
    """중국어 문장 패턴(한자 연속 + 중국어 문법 조사)이 섞였는지 확인."""
    return bool(re.search(r'[一-鿿]{4,}', text)) and bool(re.search(r'[的了是在您]', text))

# build_citation_suffix(근거 표기 조립)는 data/faq_units.py (셀 2에서 import)

def finalize_answer(raw_answer: str, unit: dict, refs: list[dict]) -> tuple[str, list[dict]]:
    """모델이 쓴 '근거:' 부분은 버리고, 답변이 실제로 쓴 참조만 골라 근거를 조립한다.
    예전에는 참조 전체를 붙여 답변에 쓰이지 않은 조문까지 근거로 나열됐다(검수: 참조 붙은 42건 중 36건)."""
    body = re.sub(r'\s*근거\s*:.*$', '', raw_answer, flags=re.DOTALL).strip()
    # 앞 조에 붙인 준용규정도 답변이 그 문장을 썼으면 근거로 표기한다
    used = rag_context.used_references(body, refs + unit.get("merged_refs", []), unit["content"],
                                       unit_article=unit.get("article_num"))
    return f"{body} {build_citation_suffix(unit, used)}", used

def fix_missing_commas(raw: str) -> str:
    """문자열 끝(") 다음에 줄바꿈 후 "키": 가 오는데 쉼표가 없으면 추가."""
    return re.sub(r'"\s*\n(\s*)"', '",\n\\1"', raw)

def extract_json(raw: str):
    raw = raw.strip()
    raw = re.sub(r"^```(json)?", "", raw)
    raw = re.sub(r"```$", "", raw)
    raw = fix_missing_commas(raw.strip())
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            return [parsed]
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass
    return None

def call_ollama_with_retry(prompt: str, unit: dict, temperature: float = 0.5, max_retries: int = 2, system: str | None = None):
    """(파싱 결과 또는 None, 원본 출력 목록) — 실패한 출력도 원인 분석용으로 돌려준다"""
    raws = []
    for _ in range(max_retries):
        raw = call_ollama(prompt, unit, temperature=temperature, system=system)
        raws.append(raw)
        parsed = extract_json(raw)
        if parsed:
            return parsed, raws
    return None, raws

def pick_system_prompt(unit: dict, version: str, topic_mode: str | None, kind: str, topics: list[str]) -> str | None:
    """None = 기존 V1(call_ollama가 유형별로 고름)"""
    if kind == "topic":
        return SYSTEM_PROMPT_TOPIC_V2C if version == "v2C" else SYSTEM_PROMPT_TOPIC
    if version == "v2C":
        assert not topics, "V2C에는 안 A(세 번째 질문 주제 지정) 프롬프트가 없다"
        return SYSTEM_PROMPT_ARTICLE_V2C if get_prompt_type(unit) == "조항형" else SYSTEM_PROMPT_GUIDE_V2C
    if version != "v2":
        return None
    if get_prompt_type(unit) != "조항형":
        return SYSTEM_PROMPT_GUIDE_V2
    return SYSTEM_PROMPT_ARTICLE_V2_TOPIC if topics else SYSTEM_PROMPT_ARTICLE_V2

def generate_for_unit(unit: dict, species: str, temperature: float = 0.5,
                      version: str = "v1", topic_mode: str | None = None, kind: str = "normal") -> dict:
    """단위 하나 생성. 반환 {"status", "faqs", "rejected", "raw_outputs", "unit_id"}
    - status: "ok" / "parse_fail"(재시도까지 JSON 실패) / "all_rejected"(QA가 전부 검증 탈락)
    - rejected: 계산 검증(faq_verify) 탈락·중국어 섞임 등으로 버린 QA와 사유 — 실패 원인을 남겨 둔다
    - version "v2": 질문 프롬프트 V2(셀 5), "v2C": V2 + 한 질문 한 가지·설정 지어내기 금지(실험 C-2). topic_mode "A": 조항형 세 번째 질문을 [주제]로. kind "topic": 주제 전용 회차(안 B)
      주제는 원문이 다루는 것만(faq_style.eligible_topics) — 원문에 없는 주제를 시키면 지어내기를 부른다"""
    refs = get_references_for_unit(unit, species)
    topics = []
    if kind == "topic" or (topic_mode == "A" and get_prompt_type(unit) == "조항형"):
        topics = faq_style.eligible_topics(unit["content"])
    assert kind != "topic" or topics, f"주제 회차인데 원문이 다루는 주제가 없음: {unit_id(unit)}"
    system = pick_system_prompt(unit, version, topic_mode, kind, topics)
    parsed, raws = call_ollama_with_retry(build_single_prompt(unit, refs, version=version, topics=topics or None), unit,
                                          temperature=temperature, system=system)
    result = {"status": "parse_fail", "faqs": [], "rejected": [], "raw_outputs": raws, "unit_id": unit_id(unit)}
    if not parsed:
        return result
    for qa in parsed:
        if not isinstance(qa, dict) or "question" not in qa or "answer" not in qa:
            result["rejected"].append({"qa": qa, "why": ["형식(question/answer 없음)"]})
            continue
        if has_chinese_sentence(qa["answer"]) or has_chinese_sentence(qa["question"]):
            result["rejected"].append({"qa": qa, "why": ["중국어 문장 섞임"]})
            continue
        # 의료비 특약 계산형 QA는 limit_table로 다시 계산해 틀리면 버린다
        check = faq_verify.verify(qa, unit.get("coverage_type"), unit.get("section_name")) \
            or faq_verify.verify_byeopyo(qa, unit.get("section_name"))      # 별표3 변호사비용·별표4 인지액
        if check is not None and not check[0]:
            result["rejected"].append({"qa": qa, "why": check[1]})
            continue
        answer, used = finalize_answer(qa["answer"], unit, refs)
        result["faqs"].append({
            "species": species, "section": unit["section"], "section_name": unit.get("section_name"),
            "article_num": unit.get("article_num"), "article_title": unit.get("article_title"),
            "coverage_type": unit.get("coverage_type"),
            "question": qa["question"], "answer": answer,
            "references": [r["citation_label"] for r in used],            # 답변이 쓴 참조(근거 표기와 같음)
            "references_given": [r["citation_label"] for r in refs],      # 프롬프트에 준 참조 전체
            "calc_verified": check is not None,
            "unit_id": unit_id(unit),
            "prompt_version": version, "gen_kind": kind, "topics": topics,     # 실험 C 구분용(06은 그대로 보존)
        })
    result["status"] = "ok" if result["faqs"] else "all_rejected"
    return result

def generate_faq_round(units, species, save_path, temperature=0.5, save_every=50, **gen_opts):
    """한 라운드: 전체 단위를 순차로 순회하며 FAQ 생성. gen_opts: generate_for_unit의 version·topic_mode·kind"""
    results, failures = [], []
    start_time = time.time()
    for i, unit in enumerate(units):
        try:
            res = generate_for_unit(unit, species, temperature, **gen_opts)
            if res["rejected"] or res["status"] != "ok":
                failures.append({"index": i, **{k: res[k] for k in ("unit_id", "status", "rejected", "raw_outputs")}})
            if res["status"] != "ok":
                print(f"[{i}] {res['status']}, 건너뜀")
                continue
            results.extend(res["faqs"])
        except Exception as e:
            print(f"[{i}] 에러: {e}")
            failures.append({"index": i, "unit_id": unit_id(unit), "status": "error", "error": str(e)})
            continue
        if (i + 1) % save_every == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"{i+1}/{len(units)} 처리, 누적 {len(results)}개, 경과 {(time.time()-start_time)/60:.1f}분")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    with open(save_path.with_name(save_path.stem + "_failures.json"), "w", encoding="utf-8") as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)       # 파싱 실패·검증 탈락 원본 보관
    elapsed = time.time() - start_time
    print(f"완료: 총 {len(results)}개, 실패·탈락 기록 {len(failures)}건 (총 {elapsed/60:.1f}분, 단위당 {elapsed/max(len(units),1):.1f}초)")
    return results


In [8]:
# ===== 8: 검수용 샘플 100개 (섹션별 할당, 단위당 1개) =====
# 전량 생성 전에 사람이 직접 검수한다. 범주별 할당 수만큼 서로 다른 단위를 골라(섹션명 라운드로빈 → 고르게)
# 단위마다 FAQ를 생성하고 그중 1개를 샘플로 둔다. 강아지/고양이는 번갈아 뽑고, 두 종에서 같은 원문은 한 번만.
# 법규는 EXCLUDE_FROM_FAQ로 뺐다 — 그 몫 10을 상담 빈도가 높은 보통약관·의료비 특약에 5씩 (2026-10-12)
SAMPLE_QUOTA = {
    "보통약관": 25, "의료비 특약": 30, "기타 특약": 10, "기타_제도성": 15,
    "별표·별첨": 10, "안내류": 10,
}

def sample_category(unit: dict) -> str:
    sec = unit["section"]
    if sec == "특별약관":
        return "의료비 특약" if "의료비" in (unit.get("section_name") or "") else "기타 특약"
    if sec == "기타_제도성_특별약관":
        return "기타_제도성"
    if sec in ("별표", "별첨"):
        return "별표·별첨"
    if sec.startswith("안내") or sec == "약관_이용":
        return "안내류"
    return sec

def _species_free(text: str) -> str:
    return re.sub(r"반려[견묘]|강아지|고양이|Puppy|Cat", "", text or "")

def pick_sample_units(seed: int = 42):
    rng = random.Random(seed)
    pools = {}
    for sp, units in [("강아지", units_dog), ("고양이", units_cat)]:
        for u in units:
            pools.setdefault(sample_category(u), {}).setdefault(sp, []).append(u)
    for cat in pools.values():
        for lst in cat.values():
            rng.shuffle(lst)
    picks, seen = {}, set()
    for cat, quota in SAMPLE_QUOTA.items():
        # 종 번갈아 → 그 종 안에서 섹션명 라운드로빈으로 다음 후보
        queues = {sp: {} for sp in pools[cat]}
        for sp, lst in pools[cat].items():
            for u in lst:
                queues[sp].setdefault(u.get("section_name"), []).append(u)
        order = {sp: list(q.keys()) for sp, q in queues.items()}
        cursor = {sp: 0 for sp in queues}
        chosen, sp_cycle = [], [sp for sp in ("강아지", "고양이") if sp in queues]
        k = 0
        while len(chosen) < quota * 3 and any(queues[sp][n] for sp in sp_cycle for n in order[sp]):
            sp = sp_cycle[k % len(sp_cycle)]; k += 1
            names = [n for n in order[sp] if queues[sp][n]]
            if not names:
                continue
            name = names[cursor[sp] % len(names)]; cursor[sp] += 1
            u = queues[sp][name].pop()
            key = (_species_free(u.get("section_name")), u.get("article_num"), u.get("coverage_type"),
                   _species_free(u["content"]))
            if key in seen:
                continue
            seen.add(key)
            chosen.append((sp, u))
        picks[cat] = chosen          # 할당의 3배까지 후보 — 파싱 실패 시 다음 후보로 대체
    return picks

sample_candidates = pick_sample_units()
for cat, quota in SAMPLE_QUOTA.items():
    print(f"{cat}: 할당 {quota} / 후보 {len(sample_candidates[cat])}")


보통약관: 할당 25 / 후보 64
의료비 특약: 할당 30 / 후보 90
기타 특약: 할당 10 / 후보 30
기타_제도성: 할당 15 / 후보 44
별표·별첨: 할당 10 / 후보 30
안내류: 할당 10 / 후보 27


In [9]:
# 샘플 생성 실행 (Ollama, temperature 0.5) — 결과: data/faq/review/faq_sample_100.json (+ 원본 출력 전체 _raw.json)
# RUN_SAMPLE = False가 기본 — 이 파일은 검수 3회차 정답지다(06 심판 시험에 씀). 전체 실행으로 덮어쓰지 않게 막는다.
# 새 검수 회차를 만들 때만 True로 바꾸고, 그 전에 기존 파일을 review/round3/ 등으로 옮겨 둘 것.
RUN_SAMPLE = False
REVIEW_DIR = FAQ_DIR / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

def run_sample_generation(temperature: float = 0.5):
    samples, raw_all, fails = [], [], 0
    start = time.time()
    for cat, quota in SAMPLE_QUOTA.items():
        got = 0
        for sp, unit in sample_candidates[cat]:
            if got >= quota:
                break
            res = generate_for_unit(unit, sp, temperature)
            if res["status"] != "ok":
                fails += 1
                raw_all.append({"id": None, "category": cat, "species": sp, **res})   # 실패 출력도 남긴다
                continue
            faqs = res["faqs"]
            pick = faqs[len(samples) % len(faqs)]       # 단위의 1·2·3번째 질문이 고르게 섞이도록
            got += 1
            samples.append({
                "id": len(samples) + 1, "category": cat, **pick,
                "context_prefix": unit.get("context_prefix"),
                "source_text": rag_context.display_text(unit),
            })
            raw_all.append({"id": len(samples), "category": cat, "species": sp, **res})
        print(f"{cat}: {got}/{quota} (누적 {len(samples)}, 실패 {fails}, {(time.time()-start)/60:.1f}분)")
    with open(REVIEW_DIR / "faq_sample_100.json", "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    with open(REVIEW_DIR / "faq_sample_100_raw.json", "w", encoding="utf-8") as f:
        json.dump(raw_all, f, ensure_ascii=False, indent=2)
    return samples, fails

if RUN_SAMPLE:
    faq_samples, sample_fails = run_sample_generation()
    n_rejected = sum(len(r["rejected"]) for r in json.load(open(REVIEW_DIR / "faq_sample_100_raw.json", encoding="utf-8")))
    print(f"샘플 {len(faq_samples)}개 저장 | 실패로 대체된 단위 {sample_fails}개 | 검증 탈락 QA {n_rejected}개 | 입력 예산 도달 {len(truncation_log)}건")
else:
    print("샘플 생성 건너뜀(RUN_SAMPLE=False) — 검수 3회차 파일 유지:", REVIEW_DIR / "faq_sample_100.json")


샘플 생성 건너뜀(RUN_SAMPLE=False) — 검수 3회차 파일 유지: /home/kopo17/AI_programming/petmint-rag-chatbot/data/faq/review/faq_sample_100.json


In [10]:
# ===== 9: 전체 라운드 생성 (샘플 검수 통과 후 실행) =====
# 라운드마다 temperature를 달리해 같은 단위에서 다른 질문을 얻는다. 라운드별 파일로 저장한다.
random.seed(42)
units_shuffled = {"강아지": units_dog.copy(), "고양이": units_cat.copy()}
for lst in units_shuffled.values():
    random.shuffle(lst)

# 생성 라운드 8~10(강아지)·9~10(고양이)은 2026-10-13 추가. 첫 전량 생성 측정: 후반 라운드 중복률이 0.35~0.4에서 39~40%,
# 0.6~0.7에서 28~30%였고 심판 통과율은 0.6~0.7이 52~57%로 ≤0.55(54~60%)보다 약 3%p 낮았다 → 0.65~0.7. 0.7 초과는 측정한 적 없어 쓰지 않는다.
# 단위 목록에서 도입 문장 단위 1개가 빠져(faq_units.is_intro_only) 섞은 순서가 1~7/1~8과 다르다 — 처리 순서만 다르고 결과에는 영향 없음.
ROUND_TEMPERATURES = {
    "강아지": [0.5, 0.45, 0.55, 0.65, 0.5, 0.6, 0.4, 0.7, 0.65, 0.7],
    "고양이": [0.5, 0.45, 0.55, 0.65, 0.6, 0.4, 0.35, 0.7, 0.65, 0.7],
}
# 추가 생성 라운드는 원문 600자 이상 단위만 돈다(2026-10-13). 첫 06 결과: 단위당 최종 QA 중앙값이 길이와 무관하게 7~11건이라
# 원문 1000자당 최종 QA가 150자 미만 단위 91건, 1000~1500자 단위 6건으로 15배 차이 — 짧은 단위는 이미 넘치고 긴 단위가 모자란다.
LONG_UNIT_ONLY_FROM_ROUND = {"강아지": 8, "고양이": 9}
LONG_UNIT_MIN_CHARS = 600

def round_done(save_path) -> bool:
    """생성 라운드가 끝까지 돌았는가. 라운드 파일은 50단위마다 중간 저장되므로 파일만으로는 알 수 없다 —
    generate_faq_round가 마지막에만 쓰는 _failures.json을 완료 표시로 본다."""
    return save_path.exists() and save_path.with_name(save_path.stem + "_failures.json").exists()

def generate_all_rounds(species: str):
    """끝난 생성 라운드는 건너뛴다 — 중간에 끊겨도 다시 부르면 끊긴 생성 라운드부터 이어 간다
    (끊긴 생성 라운드 자체는 처음부터 다시 만든다)."""
    for r, temp in enumerate(ROUND_TEMPERATURES[species], start=1):
        save_path = FAQ_DIR / f"petmint_{species}_faq_round{r}.json"
        if round_done(save_path):
            print(f"=== {species} 생성 라운드 {r}: 이미 완료 — 건너뜀 ({save_path.name})")
            continue
        if save_path.exists():
            print(f"=== {species} 생성 라운드 {r}: 중간 저장본만 있음 — 처음부터 다시 생성")
        print(f"\n=== {species} 생성 라운드 {r}/{len(ROUND_TEMPERATURES[species])} (temperature={temp}) ===")
        round_units = units_shuffled[species]
        if r >= LONG_UNIT_ONLY_FROM_ROUND[species]:
            round_units = [u for u in round_units if len(u["content"]) >= LONG_UNIT_MIN_CHARS]
            print(f"원문 {LONG_UNIT_MIN_CHARS}자 이상 단위만: {len(round_units)}/{len(units_shuffled[species])}")
        generate_faq_round(round_units, species, save_path, temperature=temp)

generate_all_rounds("강아지")
generate_all_rounds("고양이")


=== 강아지 생성 라운드 1: 이미 완료 — 건너뜀 (petmint_강아지_faq_round1.json)
=== 강아지 생성 라운드 2: 이미 완료 — 건너뜀 (petmint_강아지_faq_round2.json)
=== 강아지 생성 라운드 3: 이미 완료 — 건너뜀 (petmint_강아지_faq_round3.json)
=== 강아지 생성 라운드 4: 이미 완료 — 건너뜀 (petmint_강아지_faq_round4.json)
=== 강아지 생성 라운드 5: 이미 완료 — 건너뜀 (petmint_강아지_faq_round5.json)
=== 강아지 생성 라운드 6: 이미 완료 — 건너뜀 (petmint_강아지_faq_round6.json)
=== 강아지 생성 라운드 7: 이미 완료 — 건너뜀 (petmint_강아지_faq_round7.json)

=== 강아지 생성 라운드 8/10 (temperature=0.7) ===
원문 600자 이상 단위만: 188/301
[7] parse_fail, 건너뜀
[9] parse_fail, 건너뜀
[42] parse_fail, 건너뜀
50/188 처리, 누적 123개, 경과 2.8분
100/188 처리, 누적 253개, 경과 4.9분
[103] all_rejected, 건너뜀
[108] all_rejected, 건너뜀
[118] parse_fail, 건너뜀
[131] all_rejected, 건너뜀
[137] all_rejected, 건너뜀
[148] parse_fail, 건너뜀
150/188 처리, 누적 368개, 경과 7.8분
완료: 총 474개, 실패·탈락 기록 16건 (총 9.7분)

=== 강아지 생성 라운드 9/10 (temperature=0.65) ===
원문 600자 이상 단위만: 188/301
[25] all_rejected, 건너뜀
[45] parse_fail, 건너뜀
50/188 처리, 누적 125개, 경과 2.5분
[55] all_rejected, 건너뜀
[94] parse_fail, 건너뜀
100/188 

In [8]:
def round_done(save_path) -> bool:
    return save_path.exists() and save_path.with_name(save_path.stem + "_failures.json").exists()

In [9]:
# ===== 10: 실험 C — 질문 프롬프트 V2 소규모 검증 (2026-10-14) =====
# 전량 재생성 전에 같은 단위 28개로 기존(v1)과 새 프롬프트(V2, 주제 지정 안 A·B)를 1라운드씩 생성해 비교한다.
# 판단 기준: 06 심판 통과율이 v1과 비슷하면서 질문 성질(라벨 누출↓·상황형↑·주제)이 바뀌면 전량 재생성.
#           통과율이 크게 떨어지면 프롬프트를 더 손본다 — 특히 구어체 질문이 clause_mix를 늘리는지 본다.
# 출력: data/faq/exp_c/{설정}/petmint_강아지_faq_round{N}.json (+ _failures.json, prompt_tokens.json) — 본 라운드 파일과 분리.
# 이후: 06을 EXPERIMENT="exp_c/v1"·"exp_c/v2A"·"exp_c/v2B"로 한 번씩 → python -m data.faq_style data/faq/exp_c/v1 data/faq/exp_c/v2A data/faq/exp_c/v2B
# 실험 C-2(V2C): 완료된 설정(v1·v2A·v2B)은 round_done으로 건너뛰고 v2C·v2C-T만 생성한다 — 기존 결과는 지우지 않는다(비교 대상).
#   06을 EXPERIMENT="exp_c/v2C"·"exp_c/v2C-T"로 한 번씩 → python -m data.faq_style data/faq/exp_c/v1 data/faq/exp_c/v2A data/faq/exp_c/v2B data/faq/exp_c/v2C data/faq/exp_c/v2C-T
from collections import Counter, defaultdict

RUN_EXP_C = True         # 기본은 단위 선정·프롬프트 길이 확인만. GPU가 비었을 때 True
EXP_DIR = FAQ_DIR / "exp_c"
EXP_SPECIES = "강아지"     # 고양이 약관은 같은 조문 구조라 한 종으로 충분하다(시간 절반)
EXP_TEMPERATURE = 0.5      # 기존 생성 라운드 1과 같은 값 — faq_style 리포트의 '기존 전량 r1' 참고 행과 조건을 맞춘다

GENERAL = "반려동물 비용손해 관련 특별약관 일반조항"
MED_1_SECTIONS = {"펫퍼민트 반려견 통원의료비보장 특별약관", "펫퍼민트 반려견 통원의료비Ⅲ보장 특별약관",
                  "펫퍼민트 반려견 입원의료비Ⅱ보장 특별약관", "펫퍼민트 반려견 입원의료비Ⅲ보장 특별약관"}
# (선정 사유, 조건) — 순서대로 처음 맞는 사유를 기록한다
SELECTION_RULES = [
    ("×3 일반조항 제2·4·5·15·16조 — 가입 나이(반려동물 정의)·청구 서류·지급 기한·재가입·보장개시",
     lambda u: u.get("section_name") == GENERAL and u.get("article_num") in (2, 4, 5, 15, 16)),
    ("×3 의료비 제1조 기본형 — KNOWN_ISSUES 단서 독해 측정과 같은 묶음(통원Ⅰ·통원Ⅲ·입원Ⅱ·입원Ⅲ), 자기부담금·면책",
     lambda u: u.get("article_num") == 1 and u.get("coverage_type") == "기본형" and u.get("section_name") in MED_1_SECTIONS),
    ("×3 의료비 제2조 통원Ⅰ — 보장 제외·기왕증",
     lambda u: u.get("article_num") == 2 and u.get("section_name") == "펫퍼민트 반려견 통원의료비보장 특별약관"),
    ("안내형 — GUIDE V2 확인(섹션별 첫 단위)",
     lambda u: u.get("article_num") is None and u.get("section_name") in {"가입자 유의사항", "보험금 지급 절차"}),
    ("대조군 ×1 — 주제와 무관한 보통약관 조항(청약 철회·해약환급금)",
     lambda u: u["section"] == "보통약관" and u.get("article_num") in (20, 35)),
]

def select_exp_units(units: list[dict]) -> list[tuple[dict, str]]:
    picked, guide_seen = [], set()
    for u in units:                                    # make_units 순서(원문 순서) — 제외 조항은 이미 빠져 있다
        for reason, cond in SELECTION_RULES:
            if cond(u):
                if u.get("article_num") is None:       # 안내형은 섹션마다 하나만
                    if u["section_name"] in guide_seen:
                        break
                    guide_seen.add(u["section_name"])
                picked.append((u, reason))
                break
    return picked

exp_units = select_exp_units(units_dog if EXP_SPECIES == "강아지" else units_cat)
assert not any(faq_units.is_human_target(u) for u, _ in exp_units)
EXP_DIR.mkdir(parents=True, exist_ok=True)
selection = [{"unit_id": unit_id(u), "reason": reason, "prompt_type": get_prompt_type(u), "content_len": len(u["content"]),
              "eligible_topics": faq_style.eligible_topics(u["content"])} for u, reason in exp_units]
json.dump(selection, open(EXP_DIR / "selection.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"선정 {len(exp_units)}단위 → {EXP_DIR / 'selection.json'}")
for reason, n in Counter(s["reason"] for s in selection).items():
    print(f"  {n:2d}  {reason}")
for s in selection:
    print(f"  {s['unit_id']}  {s['content_len']}자  주제 {s['eligible_topics'] or '-'}")

# 프롬프트 길이(문자 수, 시스템+입력) — 토큰 실측은 생성 때 prompt_tokens_log로 한다(기존 측정 중앙 608 / p90 1556 / 최대 2933토큰)
lens = defaultdict(list)
for u, _ in exp_units:
    refs = get_references_for_unit(u, EXP_SPECIES)
    is_art = get_prompt_type(u) == "조항형"
    topics = faq_style.eligible_topics(u["content"])
    assert build_single_prompt(u, refs) == build_single_prompt(u, refs, version="v1")
    lens["v1"].append(len((SYSTEM_PROMPT_ARTICLE if is_art else SYSTEM_PROMPT_GUIDE) + build_single_prompt(u, refs)))
    lens["v2"].append(len((SYSTEM_PROMPT_ARTICLE_V2 if is_art else SYSTEM_PROMPT_GUIDE_V2) + build_single_prompt(u, refs, "v2")))
    if is_art and topics:
        lens["v2A"].append(len(SYSTEM_PROMPT_ARTICLE_V2_TOPIC + build_single_prompt(u, refs, "v2", topics)))
    if topics:
        lens["v2B 주제 회차"].append(len(SYSTEM_PROMPT_TOPIC + build_single_prompt(u, refs, "v2", topics)))
    lens["v2C"].append(len((SYSTEM_PROMPT_ARTICLE_V2C if is_art else SYSTEM_PROMPT_GUIDE_V2C) + build_single_prompt(u, refs, "v2C")))
    if topics:
        lens["v2C-T 주제 회차"].append(len(SYSTEM_PROMPT_TOPIC_V2C + build_single_prompt(u, refs, "v2C", topics)))
for k, v in lens.items():
    v = sorted(v)
    print(f"프롬프트 문자 수 {k}: {len(v)}단위, 중앙 {v[len(v)//2]} / 최대 {v[-1]}")

# 설정별 생성 라운드: v2B 라운드 1은 주제 지정 없는 V2 — v1과 비교하면 프롬프트 변경 효과만 분리된다
EXP_CONFIGS = {
    "v1":  [dict(version="v1")],
    "v2A": [dict(version="v2", topic_mode="A")],
    "v2B": [dict(version="v2"), dict(version="v2", topic_mode="B", kind="topic")],
    # 실험 C-2: v2C ↔ v2B r1, v2C-T r2 ↔ v2B r2. v2C-T r1은 v2C와 같은 설정이다 — v2B와 같은 구조(주제 회차가
    # 일반 회차와 같은 폴더에서 06 중복 검사를 받는다)를 맞추려고 다시 생성한다. v2C와의 차이는 표본 흔들림의 실측이 된다.
    "v2C":   [dict(version="v2C")],
    "v2C-T": [dict(version="v2C"), dict(version="v2C", topic_mode="B", kind="topic")],
}

def run_exp_c():
    summary = []
    for cfg, rounds in EXP_CONFIGS.items():
        d = EXP_DIR / cfg
        d.mkdir(parents=True, exist_ok=True)
        tokens_path = d / "prompt_tokens.json"
        for r, opts in enumerate(rounds, start=1):
            save_path = d / f"petmint_{EXP_SPECIES}_faq_round{r}.json"
            if round_done(save_path):
                print(f"=== {cfg} 라운드 {r}: 이미 완료 — 건너뜀")
                continue
            round_units = [u for u, _ in exp_units]
            if opts.get("kind") == "topic":
                round_units = [u for u in round_units if faq_style.eligible_topics(u["content"])]
            print(f"\n=== 실험 C {cfg} 라운드 {r}: {opts} — {len(round_units)}단위 ===")
            prompt_tokens_log.clear()
            t0 = time.time()
            qas = generate_faq_round(round_units, EXP_SPECIES, save_path, temperature=EXP_TEMPERATURE, **opts)
            minutes = (time.time() - t0) / 60
            tokens = json.load(open(tokens_path, encoding="utf-8")) if tokens_path.exists() else {}
            tokens[str(r)] = [used for _, used in prompt_tokens_log]
            json.dump(tokens, open(tokens_path, "w", encoding="utf-8"))
            fails = json.load(open(save_path.with_name(save_path.stem + "_failures.json"), encoding="utf-8"))
            tk = sorted(tokens[str(r)]) or [0]
            summary.append((cfg, r, len(round_units), len(qas), minutes, sum(f.get("status") == "parse_fail" for f in fails),
                            len(tk) - len(round_units), tk[len(tk)//2], tk[int(len(tk)*0.9)], tk[-1]))
    print("\n설정 | 라운드 | 단위 | QA | 분 | 파싱 실패 단위 | 재시도 호출 | 입력 토큰 중앙/p90/최대")
    for cfg, r, nu, nq, m, pf, retry, t50, t90, tmax in summary:
        print(f"{cfg} | {r} | {nu} | {nq} | {m:.1f} | {pf} | {retry} | {t50}/{t90}/{tmax}")
    if truncation_log:
        print("!! 예산에 닿은 호출:", truncation_log)
    # 다음 단계(06 심판)가 GPU를 쓰도록 생성 모델을 내린다
    requests.post(f"{OLLAMA_URL}/api/generate", json={"model": MODEL_NAME, "keep_alive": 0}, timeout=60)

if RUN_EXP_C:
    run_exp_c()
else:
    print("\nRUN_EXP_C = False — 생성하지 않음")

선정 28단위 → /home/kopo17/AI_programming/petmint-rag-chatbot/data/faq/exp_c/selection.json
   2  안내형 — GUIDE V2 확인(섹션별 첫 단위)
   2  대조군 ×1 — 주제와 무관한 보통약관 조항(청약 철회·해약환급금)
  11  ×3 일반조항 제2·4·5·15·16조 — 가입 나이(반려동물 정의)·청구 서류·지급 기한·재가입·보장개시
  10  ×3 의료비 제1조 기본형 — KNOWN_ISSUES 단서 독해 측정과 같은 묶음(통원Ⅰ·통원Ⅲ·입원Ⅱ·입원Ⅲ), 자기부담금·면책
   3  ×3 의료비 제2조 통원Ⅰ — 보장 제외·기왕증
  안내_절차|보험금 지급 절차|None|None|r25-29  1327자  주제 -
  안내_요약|가입자 유의사항|None|None|r39-42  905자  주제 -
  보통약관|None|20|None|r148-153  993자  주제 -
  보통약관|None|35|None|r209-212  316자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|2|None|r245-245  1697자  주제 ['가입 나이']
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|2|None|r246-246  853자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|2|None|r247-247  800자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|2|None|r248-249  1341자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|2|None|r250-250  231자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|4|None|r253-255  1404자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|5|None|r256-261  1370자  주제 -
  특별약관|반려동물 비용손해 관련 특별약관 일반조항|5|None|r262-262  144자  주제 -
  특별